# 07·02 — QLoRA: Fine-Tuning 70B Models on a Single GPU

> **Module:** 07 – Fine-Tuning  
> **Notebook:** 2 of 4  
> **Prerequisite:** `01_lora_finetuning.ipynb`  
> **Objective:** Understand how QLoRA stacks quantization on top of LoRA to make fine-tuning large models accessible on commodity hardware — and understand the engineering trade-offs at each step.

---

## 🎯 Learning Objectives

1. Explain the three innovations in QLoRA: NF4 quantization, double quantization, and paged optimizers
2. Understand how quantization works and why NF4 is better than INT8 for weights
3. Quantify the memory savings at each stage of the QLoRA stack
4. Fine-tune a model end-to-end using QLoRA with `bitsandbytes` + `peft`
5. Know when QLoRA is the right choice vs full LoRA vs full fine-tuning

---

## 1. The Memory Wall

Before QLoRA, fine-tuning a 13B+ model required extreme hardware:

```
Model                  Full FT memory     LoRA memory     QLoRA memory
─────────────────────────────────────────────────────────────────────
LLaMA-7B  (fp16)       ~112 GB            ~28 GB          ~6 GB  ✅
LLaMA-13B (fp16)       ~210 GB            ~52 GB          ~10 GB ✅
LLaMA-33B (fp16)       ~528 GB            ~132 GB         ~24 GB ✅
LLaMA-65B (fp16)       ~1 TB              ~260 GB         ~48 GB ✅

A single RTX 3090 (24GB) can fine-tune a 33B model with QLoRA!
```

**How?** QLoRA stacks three innovations:

```
┌───────────────────────────────────────────────────────────────────┐
│                     QLoRA MEMORY STACK                            │
│                                                                    │
│  Innovation 1: 4-bit NormalFloat (NF4) quantization               │
│    → Compress frozen base model weights from fp16 to 4-bit        │
│    → 75% memory reduction for base model                          │
│                                                                    │
│  Innovation 2: Double Quantization                                 │
│    → Quantize the quantization constants themselves               │
│    → Saves ~0.5 bits per parameter (small but free)               │
│                                                                    │
│  Innovation 3: Paged Optimizers                                    │
│    → NVIDIA unified memory: CPU RAM as overflow for optimizer      │
│    → Prevents OOM during gradient spikes                          │
│                                                                    │
│  + LoRA: only train small A/B matrices in fp16/bf16               │
│    → Trainable params stay in full precision for quality           │
└───────────────────────────────────────────────────────────────────┘
```

---

## 2. Understanding 4-bit Quantization

### 2.1 What is Quantization?

Neural network weights are stored as 32-bit or 16-bit floating point numbers. Quantization maps them to a smaller representation:

```
fp32:  32 bits per weight  → 4 bytes
fp16:  16 bits per weight  → 2 bytes
INT8:   8 bits per weight  → 1 byte
NF4:    4 bits per weight  → 0.5 bytes  ← QLoRA uses this
```

The challenge: how do you compress a continuous distribution into only 16 possible values (2^4 = 16) without losing too much information?

### 2.2 Why NF4 is Special

Pre-trained weights follow a **zero-centered normal distribution**. NF4 exploits this by placing quantization levels at quantile boundaries of the normal distribution — denser where weights cluster, sparser in the tails:

```
Uniform INT4 quantization:      |-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|
                                 equal spacing regardless of density

NF4 (quantile-based):           |--|-|--|--|--|--|--|--|--|--|--|
                                 denser near zero where weights cluster

Result: NF4 has lower quantization error for normally-distributed weights
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install transformers peft datasets trl bitsandbytes accelerate matplotlib scipy -q

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple
import math
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Implementing NF4 Quantization From Scratch

Understanding the internals makes you a better engineer — you'll know exactly what `bitsandbytes` does under the hood.

In [ ]:
class NF4Quantizer:
    """
    NormalFloat-4 quantization from Dettmers et al. (2023).

    NF4 places 16 quantization levels at the quantiles of a standard normal
    distribution, optimally representing normally-distributed weights.

    This is an educational implementation — bitsandbytes uses CUDA kernels
    for the actual computation.
    """

    # Pre-computed NF4 levels: 16 values at quantiles of N(0,1)
    # These are the exact values from the QLoRA paper appendix
    NF4_LEVELS = torch.tensor([
        -1.0000, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911,
         0.0000,
         0.0796,  0.1609,  0.2461,  0.3379,  0.4407,  0.5626,  0.7230,
         1.0000
    ])

    def __init__(self, block_size: int = 64):
        """
        block_size: number of weights per quantization block.
        Each block has its own scaling constant for better accuracy.
        Typical values: 32, 64, 128.
        """
        self.block_size = block_size
        self.levels = self.NF4_LEVELS

    def quantize(self, weight: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Quantize a weight tensor to NF4.

        Returns:
            quantized: int tensor of indices (0-15) into NF4_LEVELS
            absmax:    per-block absmax scaling constants (fp32)
        """
        original_shape = weight.shape
        flat = weight.float().flatten()

        # Pad to multiple of block_size
        pad_len = (self.block_size - len(flat) % self.block_size) % self.block_size
        if pad_len > 0:
            flat = torch.cat([flat, torch.zeros(pad_len)])

        # Reshape into blocks
        n_blocks = len(flat) // self.block_size
        blocks = flat.view(n_blocks, self.block_size)

        # Per-block normalization: scale each block to [-1, 1]
        absmax = blocks.abs().max(dim=1).values.clamp(min=1e-8)  # (n_blocks,)
        normalized = blocks / absmax.unsqueeze(1)  # each block in [-1, 1]

        # Find nearest NF4 level for each weight
        flat_norm = normalized.flatten()
        # Vectorized nearest-neighbor search
        diffs = (flat_norm.unsqueeze(1) - self.levels.unsqueeze(0)).abs()  # (N, 16)
        indices = diffs.argmin(dim=1).to(torch.uint8)  # (N,)

        return indices[:len(weight.flatten())], absmax

    def dequantize(self, indices: torch.Tensor, absmax: torch.Tensor,
                   original_shape: torch.Size) -> torch.Tensor:
        """
        Reconstruct approximate fp32 weights from NF4 indices.
        """
        # Pad indices to block multiple
        n_elements = len(indices)
        pad_len = (self.block_size - n_elements % self.block_size) % self.block_size
        if pad_len > 0:
            indices = torch.cat([indices, torch.zeros(pad_len, dtype=torch.uint8)])

        n_blocks = len(indices) // self.block_size
        values = self.levels[indices.long()]  # map indices → NF4 levels
        blocks = values.view(n_blocks, self.block_size)

        # Rescale each block
        dequantized = (blocks * absmax.unsqueeze(1)).flatten()
        return dequantized[:n_elements].view(original_shape)

    def quantize_error(self, weight: torch.Tensor) -> Dict:
        """Measure quantization error (MSE and max error)."""
        indices, absmax = self.quantize(weight)
        reconstructed = self.dequantize(indices, absmax, weight.shape)
        error = (weight.float() - reconstructed).abs()
        return {
            'mse':      float(error.pow(2).mean()),
            'mae':      float(error.mean()),
            'max_error': float(error.max()),
            'relative_mse': float(error.pow(2).mean() / weight.float().pow(2).mean().clamp(1e-8)),
        }


# Compare NF4 vs uniform INT4 quantization error
class UniformINT4Quantizer:
    """Simple uniform 4-bit quantization for comparison."""
    def quantize_error(self, weight: torch.Tensor) -> Dict:
        w = weight.float()
        w_min, w_max = w.min(), w.max()
        scale = (w_max - w_min) / 15  # 16 levels: 0..15
        quantized = torch.clamp(((w - w_min) / scale.clamp(1e-8)).round(), 0, 15)
        reconstructed = quantized * scale + w_min
        error = (w - reconstructed).abs()
        return {
            'mse':      float(error.pow(2).mean()),
            'mae':      float(error.mean()),
            'max_error': float(error.max()),
            'relative_mse': float(error.pow(2).mean() / w.pow(2).mean().clamp(1e-8)),
        }


torch.manual_seed(42)
nf4 = NF4Quantizer(block_size=64)
int4 = UniformINT4Quantizer()

# Test on realistic weight distributions
print("Quantization Error Comparison (NF4 vs Uniform INT4)")
print(f"{'Distribution':<28} {'NF4 MSE':>10} {'INT4 MSE':>10} {'NF4 wins?':>12}")
print("-" * 65)

distributions = [
    ("Normal N(0, 0.02)",    torch.randn(1024, 1024) * 0.02),
    ("Normal N(0, 0.1)",     torch.randn(1024, 1024) * 0.1),
    ("Normal N(0, 1.0)",     torch.randn(1024, 1024)),
    ("Uniform [-1, 1]",      torch.rand(1024, 1024) * 2 - 1),
    ("Laplace(0, 0.1)",      torch.distributions.Laplace(0, 0.1).sample((1024, 1024))),
]

for name, weights in distributions:
    e_nf4 = nf4.quantize_error(weights)
    e_int4 = int4.quantize_error(weights)
    nf4_better = e_nf4['mse'] < e_int4['mse']
    print(f"{name:<28} {e_nf4['mse']:>10.2e} {e_int4['mse']:>10.2e} "
          f"{'✅ Yes' if nf4_better else '❌ No':>12}")

print()
print("NF4 wins for normally-distributed weights (which pre-trained weights follow).")
print("Uniform INT4 wins for uniformly-distributed weights (which activations can have).")
print("This is why QLoRA uses NF4 for weights but keeps activations in bf16.")

In [ ]:
# Visualize quantization: where NF4 levels fall on a normal distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
x = np.linspace(-3.5, 3.5, 1000)
y = stats.norm.pdf(x, 0, 1)
ax.fill_between(x, y, alpha=0.25, color='steelblue', label='N(0,1) distribution')
ax.plot(x, y, 'steelblue', linewidth=2)

nf4_levels = nf4.levels.numpy()
int4_levels = np.linspace(-1, 1, 16)

for lvl in nf4_levels:
    ax.axvline(x=lvl, color='#d62728', alpha=0.7, linewidth=1.2)
for lvl in int4_levels:
    ax.axvline(x=lvl, color='#2ca02c', alpha=0.5, linewidth=1.2, linestyle='--')

ax.set_xlabel('Weight Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('NF4 vs Uniform INT4 Quantization Levels\non Normal Distribution', fontsize=12)
legend_elements = [
    mpatches.Patch(color='#d62728', label='NF4 levels (dense near zero)'),
    mpatches.Patch(color='#2ca02c', label='INT4 levels (uniform spacing)'),
]
ax.legend(handles=legend_elements)
ax.grid(alpha=0.3)
ax.set_xlim(-3.5, 3.5)

ax = axes[1]
# Reconstruction error as function of weight value
test_values = torch.linspace(-2, 2, 500)
test_weights = test_values.unsqueeze(0).unsqueeze(0)  # (1,1,500)

# NF4 nearest-level error at each point
diffs_nf4 = (test_values.unsqueeze(1) - nf4.levels.unsqueeze(0)).abs()
nf4_nearest = nf4.levels[diffs_nf4.argmin(dim=1)]
nf4_error = (test_values - nf4_nearest).abs().numpy()

int4_nearest = torch.tensor([int4_levels[np.abs(int4_levels - v.item()).argmin()]
                               for v in test_values])
int4_error = (test_values - int4_nearest).abs().numpy()

ax.plot(test_values.numpy(), nf4_error, color='#d62728', label='NF4 error', linewidth=2)
ax.plot(test_values.numpy(), int4_error, color='#2ca02c', label='INT4 error',
        linewidth=2, linestyle='--')

# Shade density to show where weights concentrate
density_x = np.linspace(-2, 2, 500)
density_y = stats.norm.pdf(density_x, 0, 0.8) * 0.05
ax.fill_between(density_x, 0, density_y, alpha=0.15, color='steelblue',
                label='Weight density')

ax.set_xlabel('Weight Value', fontsize=12)
ax.set_ylabel('Absolute Quantization Error', fontsize=12)
ax.set_title('Per-value Quantization Error: NF4 vs INT4', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/07_nf4_vs_int4.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"NF4 mean error on N(0,0.8): {nf4_error[125:375].mean():.4f}")
print(f"INT4 mean error on N(0,0.8): {int4_error[125:375].mean():.4f}")
print(f"NF4 advantage: {(int4_error[125:375].mean() / nf4_error[125:375].mean()):.2f}x lower error")

## 4. Double Quantization: Compressing the Compressor

In [ ]:
def memory_breakdown(n_params: int, block_size: int = 64) -> Dict:
    """
    Compute memory usage at each stage of QLoRA quantization.

    Demonstrates the Double Quantization savings.
    """
    n_blocks = math.ceil(n_params / block_size)

    bytes_fp32 = n_params * 4
    bytes_fp16 = n_params * 2

    # NF4: 4 bits per weight = 0.5 bytes, PLUS fp32 absmax per block
    bytes_nf4_weights = n_params * 0.5
    bytes_nf4_constants_fp32 = n_blocks * 4          # 4 bytes per block constant
    bytes_nf4_total = bytes_nf4_weights + bytes_nf4_constants_fp32

    # Double quantization: quantize the absmax constants themselves to fp8
    # Then store one fp32 super-constant per 256 block constants
    bytes_dq_block_constants = n_blocks * 1           # fp8 ≈ 1 byte
    bytes_dq_super_constants = math.ceil(n_blocks / 256) * 4  # fp32 super-constant
    bytes_dq_total = bytes_nf4_weights + bytes_dq_block_constants + bytes_dq_super_constants

    return {
        'fp32_GB':          bytes_fp32 / 1e9,
        'fp16_GB':          bytes_fp16 / 1e9,
        'nf4_GB':           bytes_nf4_total / 1e9,
        'nf4_dq_GB':        bytes_dq_total / 1e9,
        'nf4_vs_fp16':      bytes_fp16 / bytes_nf4_total,
        'dq_saving_MB':     (bytes_nf4_total - bytes_dq_total) / 1e6,
        'bits_per_param_nf4':    bytes_nf4_total * 8 / n_params,
        'bits_per_param_dq':     bytes_dq_total * 8 / n_params,
        'n_params':         n_params,
        'n_blocks':         n_blocks,
    }


# Real model parameter counts
models = [
    ("LLaMA-3.1-8B",   8_030_261_248),
    ("LLaMA-3.1-13B",  13_015_864_320),
    ("LLaMA-3.1-70B",  70_553_706_496),
]

print(f"{'Model':<18} {'fp16':>8} {'NF4':>8} {'NF4+DQ':>8} "
      f"{'Savings vs fp16':>16} {'DQ saving':>12} {'bits/param':>11}")
print("-" * 90)

for name, n_params in models:
    r = memory_breakdown(n_params)
    print(f"{name:<18} "
          f"{r['fp16_GB']:>7.1f}G "
          f"{r['nf4_GB']:>7.1f}G "
          f"{r['nf4_dq_GB']:>7.1f}G "
          f"{r['nf4_vs_fp16']:>14.2f}x "
          f"{r['dq_saving_MB']:>10.0f}MB "
          f"{r['bits_per_param_dq']:>9.2f}")

print()
print("Double quantization saves ~0.37 bits per parameter.")
print("Small per-param, but meaningful at 70B: saves ~3.2 GB!")

## 5. The Full QLoRA Stack: Memory Budget

In [ ]:
def qlora_memory_budget(
    model_params: int,
    lora_rank: int = 16,
    n_lora_layers: int = 32,
    d_model: int = 4096,
    batch_size: int = 4,
    seq_len: int = 512,
    grad_accum: int = 4,
) -> Dict:
    """
    Full memory budget breakdown for a QLoRA training run.
    Useful for deciding whether your GPU can handle a given configuration.
    """
    # ── Base model (NF4 + DQ) ────────────────────────────────────────
    n_blocks = math.ceil(model_params / 64)
    base_model_bytes = (model_params * 0.5
                        + n_blocks * 1
                        + math.ceil(n_blocks / 256) * 4)

    # ── LoRA adapters (fp16/bf16) ────────────────────────────────────
    # 2 matrices per layer: A (r×d) and B (d×r)
    lora_params = 2 * lora_rank * d_model * n_lora_layers
    lora_bytes = lora_params * 2  # bf16

    # ── LoRA gradients (same size as lora params) ────────────────────
    grad_bytes = lora_params * 2

    # ── Paged Adam optimizer states ──────────────────────────────────
    # Adam: fp32 first moment + fp32 second moment = 2 × fp32 per param
    # Paged: overflows to CPU RAM, but GPU footprint = mini-batch states
    optim_bytes = lora_params * 8  # 2 × fp32

    # ── Activations (for gradient checkpointing, store only boundaries) ─
    # Without grad checkpointing: batch × seq × d_model × layers × 2 bytes
    # With grad checkpointing: ~sqrt(layers) checkpoints
    activations_no_ckpt = batch_size * seq_len * d_model * n_lora_layers * 2
    activations_with_ckpt = batch_size * seq_len * d_model * int(math.sqrt(n_lora_layers)) * 2

    total_no_ckpt   = base_model_bytes + lora_bytes + grad_bytes + optim_bytes + activations_no_ckpt
    total_with_ckpt = base_model_bytes + lora_bytes + grad_bytes + optim_bytes + activations_with_ckpt

    return {
        'base_model_GB':           base_model_bytes / 1e9,
        'lora_params_GB':          lora_bytes / 1e9,
        'gradients_GB':            grad_bytes / 1e9,
        'optimizer_GB':            optim_bytes / 1e9,
        'activations_no_ckpt_GB':  activations_no_ckpt / 1e9,
        'activations_with_ckpt_GB': activations_with_ckpt / 1e9,
        'total_no_ckpt_GB':        total_no_ckpt / 1e9,
        'total_with_ckpt_GB':      total_with_ckpt / 1e9,
        'lora_params':             lora_params,
        'base_model_params':       model_params,
        'trainable_pct':           100 * lora_params / model_params,
    }


configs = [
    ("LLaMA-3-8B  (RTX 3090 24GB?)",  8_030_261_248,  32, 4096),
    ("LLaMA-3-13B (RTX 3090 24GB?)",  13_015_864_320, 40, 5120),
    ("LLaMA-3-70B (A100 80GB?)",      70_553_706_496, 80, 8192),
]

for name, params, layers, d in configs:
    r = qlora_memory_budget(params, lora_rank=16, n_lora_layers=layers, d_model=d,
                             batch_size=4, seq_len=512)
    print(f"\n{name}")
    print(f"  Base model (NF4+DQ):    {r['base_model_GB']:6.2f} GB")
    print(f"  LoRA adapters (bf16):   {r['lora_params_GB']:6.2f} GB  ({r['trainable_pct']:.3f}% of params)")
    print(f"  Gradients:              {r['gradients_GB']:6.2f} GB")
    print(f"  Optimizer (paged):      {r['optimizer_GB']:6.2f} GB")
    print(f"  Activations (no ckpt):  {r['activations_no_ckpt_GB']:6.2f} GB")
    print(f"  Activations (ckpt):     {r['activations_with_ckpt_GB']:6.2f} GB")
    print(f"  ─────────────────────────────")
    print(f"  TOTAL (no ckpt):        {r['total_no_ckpt_GB']:6.2f} GB  ← needs this much VRAM")
    print(f"  TOTAL (with ckpt):      {r['total_with_ckpt_GB']:6.2f} GB  ← ~30% slower but fits smaller GPU")

## 6. QLoRA in Practice: Full Training Setup

In [ ]:
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer

# ── Step 1: Configure 4-bit quantization ─────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # Load weights in 4-bit NF4
    bnb_4bit_quant_type="nf4",                  # Use NF4 (not fp4)
    bnb_4bit_compute_dtype=torch.bfloat16,      # Compute in bf16 (dequantize → compute → keep bf16)
    bnb_4bit_use_double_quant=True,             # Enable double quantization
)

# ── Step 2: Load model in 4-bit ───────────────────────────────────────────
MODEL_NAME = "gpt2"  # Small for demo. In production: "meta-llama/Llama-3.1-8B"

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Critical for Causal LM training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    # quantization_config=bnb_config,  # Uncomment for real 4-bit (requires CUDA)
    torch_dtype=torch.float32,           # Use float32 for CPU demo
    device_map="auto",
)

# ── Step 3: Prepare model for k-bit training ──────────────────────────────
# This critical step:
#   1. Casts layer norms to fp32 (stability)
#   2. Enables gradient checkpointing correctly
#   3. Upcasts the final lm_head to fp32
# model = prepare_model_for_kbit_training(model)  # Uncomment for real 4-bit

# ── Step 4: Add LoRA adapters ─────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],  # GPT-2 attention modules
    # For LLaMA: ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\nMemory estimate:")
total_params  = sum(p.numel() for p in model.parameters())
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - train_params
print(f"  Total parameters:     {total_params:>15,}")
print(f"  Trainable (LoRA A/B): {train_params:>15,}  ({100*train_params/total_params:.3f}%)")
print(f"  Frozen (base model):  {frozen_params:>15,}")

In [ ]:
# ── Step 5: Prepare dataset ───────────────────────────────────────────────
# QLoRA works best with instruction-formatted data.
# This is a tiny demo dataset — real training uses thousands of examples.

INSTRUCTION_TEMPLATE = """### Instruction:
{instruction}

### Response:
{response}"""

training_data = [
    {"instruction": "What is gradient descent?",
     "response": "Gradient descent is an optimization algorithm that iteratively moves parameters in the direction of steepest descent of the loss function. At each step it computes the gradient of the loss with respect to the parameters and subtracts a small fraction (the learning rate) of it."},
    {"instruction": "Explain the attention mechanism.",
     "response": "Attention computes a weighted sum of value vectors, where weights are determined by the similarity between a query vector and key vectors. For self-attention, queries, keys, and values all come from the same sequence. This allows each token to attend to every other token regardless of distance."},
    {"instruction": "What is overfitting?",
     "response": "Overfitting occurs when a model learns the training data too well, capturing noise and irrelevant patterns. The result is high training accuracy but poor performance on new, unseen data. Prevention strategies include regularization, dropout, early stopping, and collecting more diverse training data."},
    {"instruction": "What is the difference between batch normalization and layer normalization?",
     "response": "Batch normalization normalizes across the batch dimension — it computes statistics over all examples in a mini-batch for each feature. Layer normalization normalizes across the feature dimension — it computes statistics over all features for each individual example. LayerNorm is preferred in transformers because it works correctly with variable-length sequences and doesn't depend on batch size."},
    {"instruction": "What is the vanishing gradient problem?",
     "response": "The vanishing gradient problem occurs in deep networks during backpropagation: gradients become exponentially small as they flow backward through many layers, making early layers learn very slowly or not at all. Sigmoid and tanh activations are especially prone to this because their derivatives are always less than 1. Solutions include ReLU activations, residual connections, proper weight initialization, and normalization layers."},
] * 10  # Repeat for demo purposes

def format_example(example):
    return {"text": INSTRUCTION_TEMPLATE.format(**example)}

dataset = Dataset.from_list([format_example(d) for d in training_data])
print(f"Training examples: {len(dataset)}")
print(f"\nExample formatted input:")
print(dataset[0]['text'][:300] + "...")

In [ ]:
# ── Step 6: Training arguments ────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./qlora_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # Effective batch size = 8
    gradient_checkpointing=True,        # Reduces activation memory ~60%
    optim="paged_adamw_32bit",          # QLoRA paged optimizer
    # optim="paged_adamw_8bit",         # Even more memory efficient
    learning_rate=2e-4,                 # Higher LR works for LoRA
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_grad_norm=0.3,                  # Lower for QLoRA stability
    fp16=False,                         # Use bf16 if your GPU supports it
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    dataloader_num_workers=0,
)

# ── Step 7: SFT Trainer ───────────────────────────────────────────────────
# SFTTrainer (Supervised Fine-Tuning) handles:
#   - Data formatting and packing
#   - Proper loss masking (don't compute loss on prompt tokens)
#   - Sequence packing for efficiency
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_seq_length=256,
    dataset_text_field="text",
    packing=False,  # Set True for very short examples (packs multiple into one sequence)
)

print("Trainer configured. Starting training...")
trainer.train()
print("\nTraining complete!")

In [ ]:
# ── Step 8: Save and load the adapter ────────────────────────────────────
# IMPORTANT: Save ONLY the LoRA adapter, not the full model.
# The adapter is tiny (a few MB) vs the full model (several GB).

model.save_pretrained("./qlora_adapter")
tokenizer.save_pretrained("./qlora_adapter")

adapter_size_mb = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
    if p.requires_grad
) / 1e6
print(f"\nAdapter size: {adapter_size_mb:.1f} MB")
print("(vs full model: several GB)")

# ── Inference: load adapter onto base model ───────────────────────────────
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
peft_model = PeftModel.from_pretrained(base_model, "./qlora_adapter")

# Test the fine-tuned model
def generate_response(model, tokenizer, instruction: str, max_new_tokens: int = 100) -> str:
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

test_questions = [
    "What is regularization in machine learning?",
    "Explain what a neural network is.",
]

print("Fine-tuned model responses:")
for q in test_questions:
    response = generate_response(peft_model, tokenizer, q)
    print(f"\nQ: {q}")
    print(f"A: {response[:200]}")

## 7. Hyperparameter Guide for QLoRA

### Critical Settings

In [ ]:
# Visualize the training stability impact of key hyperparameters
# (Using synthetic data to illustrate the concepts)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

steps = np.arange(1, 101)

# ── Learning rate effect ──────────────────────────────────────────────────
ax = axes[0, 0]
np.random.seed(42)
for lr, color, label in [
    (1e-3, '#d62728', 'lr=1e-3 (too high, unstable)'),
    (2e-4, '#2ca02c', 'lr=2e-4 (recommended)'),
    (1e-5, '#1f77b4', 'lr=1e-5 (too low, slow convergence)'),
]:
    # Simulate loss curves
    if lr == 1e-3:
        loss = 2.5 * np.exp(-steps/30) + 0.3 + np.random.randn(100) * 0.2
    elif lr == 2e-4:
        loss = 2.5 * np.exp(-steps/25) + 0.1 + np.abs(np.random.randn(100)) * 0.05
    else:
        loss = 2.5 * np.exp(-steps/80) + 0.4 + np.abs(np.random.randn(100)) * 0.02
    ax.plot(steps, loss, color=color, label=label, linewidth=2, alpha=0.85)

ax.set_xlabel('Steps')
ax.set_ylabel('Training Loss')
ax.set_title('Learning Rate Effect on QLoRA Training')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 3)

# ── Rank vs quality ───────────────────────────────────────────────────────
ax = axes[0, 1]
ranks = [4, 8, 16, 32, 64]
# Simulated: quality improves with rank but diminishing returns; memory increases linearly
quality = [0.61, 0.72, 0.82, 0.87, 0.89]
memory  = [0.8,  1.2,  2.0,  3.6,  7.0]   # Adapter memory in MB for 7B model

ax2 = ax.twinx()
ax.plot(ranks, quality, 'o-', color='#2ca02c', linewidth=2, markersize=8, label='Quality (eval)')
ax2.plot(ranks, memory, 's--', color='#d62728', linewidth=2, markersize=8, label='Adapter memory (MB)')
ax.set_xlabel('LoRA Rank (r)')
ax.set_ylabel('Eval Score (0-1)', color='#2ca02c')
ax2.set_ylabel('Adapter Memory (MB)', color='#d62728')
ax.set_title('LoRA Rank: Quality vs Memory Trade-off')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
ax.grid(alpha=0.3)
ax.axvline(16, color='navy', linestyle=':', alpha=0.7, label='Recommended r=16')

# ── Gradient checkpointing tradeoff ──────────────────────────────────────
ax = axes[1, 0]
seq_lens = [128, 256, 512, 1024, 2048]
mem_no_ckpt   = [x*0.012 for x in seq_lens]  # GB
mem_with_ckpt = [x*0.004 for x in seq_lens]  # ~60% savings
speed_penalty = [0, 5, 12, 22, 35]           # % slowdown

ax.bar([x - 60 for x in seq_lens], mem_no_ckpt, 100, alpha=0.8,
       color='#d62728', label='Without grad checkpointing')
ax.bar([x + 60 for x in seq_lens], mem_with_ckpt, 100, alpha=0.8,
       color='#2ca02c', label='With grad checkpointing')
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Activation Memory (GB)')
ax.set_title('Gradient Checkpointing: Memory vs Speed')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# ── Compute dtype effect ──────────────────────────────────────────────────
ax = axes[1, 1]
dtypes = ['fp32\n(baseline)', 'fp16', 'bf16', 'fp8\n(experimental)']
mem_savings = [0, 50, 50, 75]      # % memory savings vs fp32
stability   = [100, 75, 95, 70]    # relative training stability (subjective)
colors_dt = ['#aec7e8', '#1f77b4', '#2ca02c', '#ff7f0e']

x = np.arange(len(dtypes))
ax.bar(x - 0.2, mem_savings, 0.35, label='Memory saving (%)', color='#1f77b4', alpha=0.8)
ax.bar(x + 0.2, stability,   0.35, label='Stability score', color='#2ca02c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(dtypes)
ax.set_ylabel('Score / Percentage')
ax.set_title('Compute Dtype: Memory vs Stability')
ax.legend()
ax.grid(alpha=0.3, axis='y')
ax.axhline(50, color='gray', linestyle='--', alpha=0.4)

plt.suptitle('QLoRA Hyperparameter Reference Guide', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/07_qlora_hyperparams.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. When to Use QLoRA vs Other Methods

```
Decision Tree:

How much GPU memory do you have?
│
├── < 24 GB (consumer GPU)
│   └── Model size?
│       ├── ≤ 7B   → LoRA in bf16 (fits on 24GB)
│       └── > 7B   → QLoRA (NF4 + LoRA)
│
├── 24–80 GB (A10/A100 40GB)
│   └── Quality critical?
│       ├── Yes    → Full LoRA or full fine-tuning (subset of layers)
│       └── No     → QLoRA (lower cost, good quality)
│
└── > 80 GB (A100 80GB / H100)
    └── Full fine-tuning OR LoRA with high rank (r=64+)
```

### Quality Comparison (Typical Results)

| Method | Quality vs Full FT | Memory | Speed |
|--------|-------------------|--------|-------|
| Full fine-tuning | 100% (baseline) | 100% | 100% |
| LoRA (r=16, bf16) | 97-99% | ~25% | ~85% |
| QLoRA (NF4+LoRA r=16) | 94-97% | ~8% | ~65% |
| QLoRA (NF4+LoRA r=64) | 97-99% | ~12% | ~60% |

**The QLoRA paper showed 4-bit quantization + LoRA matches 16-bit LoRA on most benchmarks** — the quantization noise from NF4 is offset by the larger model you can now fit.

## 9. Engineering Notes

```python
# ✅ Critical production settings for QLoRA

# 1. Always use bfloat16 compute dtype (more stable than float16 for QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # NOT float16
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# 2. prepare_model_for_kbit_training is MANDATORY before adding LoRA
model = prepare_model_for_kbit_training(model)

# 3. Use paged optimizer to handle gradient spikes
optim="paged_adamw_32bit"   # or paged_adamw_8bit for even less memory

# 4. Enable gradient checkpointing to save activation memory
gradient_checkpointing=True

# 5. Lower max_grad_norm than standard training (stability)
max_grad_norm=0.3   # vs typical 1.0

# 6. Flash Attention 2 for sequence length > 512
model = AutoModelForCausalLM.from_pretrained(
    ...,
    attn_implementation="flash_attention_2"
)
```

## 10. Exercises

1. **NF4 vs INT4 Ablation**: Fine-tune the same model with `bnb_4bit_quant_type="nf4"` vs `"fp4"`. Measure validation loss after 1 epoch. Which is better for normally-distributed weights?

2. **Double Quantization Impact**: Measure GPU memory usage with and without `bnb_4bit_use_double_quant=True`. Does the saving match the formula from Section 4?

3. **Rank Sweep**: Train QLoRA with ranks [4, 8, 16, 32, 64] on a classification benchmark. Plot eval accuracy vs adapter size. Find the efficiency frontier.

4. **Paged Optimizer Stress Test**: Train with a large batch size until you hit an OOM spike. Compare `paged_adamw_32bit` vs `adamw_torch`. Does paging prevent the crash?

5. **Merge and Benchmark**: After QLoRA training, merge the adapter, run the model through a standard benchmark (e.g., MMLU 5-shot), and compare to the untuned base model. How much did fine-tuning help on in-domain questions?

---

## 11. References

- [Dettmers et al. (2023) — QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- [Dettmers et al. (2022) — LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale](https://arxiv.org/abs/2208.07339)
- [bitsandbytes Library](https://github.com/TimDettmers/bitsandbytes)
- [HuggingFace PEFT QLoRA Guide](https://huggingface.co/docs/peft/conceptual_guides/quantization)
- [Tim Dettmers Blog: The Case for 4-bit Precision](https://timdettmers.com/2023/01/30/which-gpu-for-deep-learning/)